In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor


from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#from sklearn.model_selection import cross_val_score, GridSearchCV

import joblib

import warnings
warnings.filterwarnings("ignore") 
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
df = pd.read_csv(r"D:\medical_cost_prediction\Main Folder\data\processed\cleaned_data.csv")
print("Shape of dataset:", df.shape)
print("Columns:", df.columns.values)
print("\nFirst 5 rows:")
display(df.head())
display(df.info())

Shape of dataset: (10212, 9)
Columns: ['age' 'sex' 'bmi' 'children' 'smoker' 'region' 'charges' 'log_charges'
 'age_bmi_interaction']

First 5 rows:


,age,sex,bmi,children,smoker,region,charges,log_charges,age_bmi_interaction
0,51,1,34.8,2,0,southwest,25557.57,10.148689,1774.8
1,50,0,30.0,2,1,northeast,39204.42,10.576545,1500.0
2,26,1,21.1,5,0,northeast,15016.92,9.616933,548.6
3,25,0,22.7,4,0,southwest,15912.13,9.674837,567.5
4,30,0,35.2,1,0,southwest,18665.40,9.834427,1056.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10212 entries, 0 to 10211
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  10212 non-null  int64  
 1   sex                  10212 non-null  int64  
 2   bmi                  10212 non-null  float64
 3   children             10212 non-null  int64  
 4   smoker               10212 non-null  int64  
 5   region               10212 non-null  object 
 6   charges              10212 non-null  float64
 7   log_charges          10212 non-null  float64
 8   age_bmi_interaction  10212 non-null  float64
dtypes: float64(4), int64(4), object(1)
memory usage: 718.2+ KB


None

### **MODEL SELECTION**
---

In [3]:
# Problem type
print(f"Problem Type: {"Regression"}")

Problem Type: Regression


In [4]:
# Separate Features & Target
X_model_selection = df.drop(["region", "charges","log_charges"], axis=1)
y_model_selection = df["charges"]

In [5]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_model_selection, y_model_selection, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8169, 6) (2043, 6) (8169,) (2043,)


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Scale numerical features
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
print("Scaled Numerical Features")

Scaled Numerical Features


In [7]:
models = {
    "Linear Regression": LinearRegression(),

    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),

    "Gradient Boosting": GradientBoostingRegressor(
                n_estimators=100, learning_rate=0.1, random_state=42),

    "XGBoost": XGBRegressor(
                n_estimators=200, learning_rate=0.05,
                max_depth=4, random_state=42)
}

In [8]:
# Training + Evaluation
results = []

for name, model in models.items():
    # Train model
    model.fit(X_train, y_train)
    
    # Predict on train & test
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)

    mae = mean_absolute_error(y_test, y_test_pred)

    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    
    # Store results
    results.append([name, mae, rmse_train, rmse_test, r2_train, r2_test])
    
    # Print results
    print(f"\n{name}")
    print("MAE:", mae)
    print("RMSE Train:", rmse_train)
    print("RMSE Test:", rmse_test)
    print("R2 Train:", r2_train)
    print("R2 Test:", r2_test)



Linear Regression
MAE: 1537.4076834334317
RMSE Train: 2003.2565045887704
RMSE Test: 1929.3773393830834
R2 Train: 0.9099187543962469
R2 Test: 0.9184495092368911

Random Forest
MAE: 1676.2907216384829
RMSE Train: 851.2714523776609
RMSE Test: 2109.17040101925
R2 Train: 0.9837333771090251
R2 Test: 0.9025424323163532

Gradient Boosting
MAE: 1537.320422313172
RMSE Train: 1946.572302230682
RMSE Test: 1935.2167834467255
R2 Train: 0.9149445123920443
R2 Test: 0.9179551215749652

XGBoost
MAE: 1545.3283270045583
RMSE Train: 1918.0522530591368
RMSE Test: 1941.8368188013703
R2 Train: 0.9174186214460147
R2 Test: 0.9173928393608293


In [9]:
# Convert to DataFrame for better comparison
results_df = pd.DataFrame(results, columns=[
    "Model", "MAE", "RMSE_Train", "RMSE_Test", "R2_Train", "R2_Test"
])

print("\nFinal Model Comparison:")
display(results_df.sort_values(by="R2_Test", ascending=False))


Final Model Comparison:


,Model,MAE,RMSE_Train,RMSE_Test,R2_Train,R2_Test
0,Linear Regression,1537.407683,2003.256505,1929.377339,0.909919,0.918450
2,Gradient Boosting,1537.320422,1946.572302,1935.216783,0.914945,0.917955
3,XGBoost,1545.328327,1918.052253,1941.836819,0.917419,0.917393
1,Random Forest,1676.290722,851.271452,2109.170401,0.983733,0.902542


In [10]:
best_model_name = results_df.iloc[0]["Model"]
print("\nBest Model:", best_model_name)


Best Model: Linear Regression


In [11]:
# Cross Validation
best_model = LinearRegression()

cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')

print("Cross Validation R2:", cv_scores.mean())

Cross Validation R2: 0.9096583875430388


### **RESULTS**

---

### **Final Pipeline**
----

In [12]:
# Separate Features & Target
X = df.drop(["region", "charges","log_charges"], axis=1)
y = df["charges"]

In [13]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(8169, 6) (2043, 6) (8169,) (2043,)


In [14]:
# Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

In [15]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None


In [16]:
y_train_pred = pipeline.predict(X_train)
y_test_pred = pipeline.predict(X_test)

print("Train Performance:")
print("R2:", r2_score(y_train, y_train_pred))

print("\nTest Performance:")
print("MAE:", mean_absolute_error(y_test, y_test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred)))
print("R2:", r2_score(y_test, y_test_pred))

Train Performance:
R2: 0.9099187543962469

Test Performance:
MAE: 1537.4076834334317
RMSE: 1929.3773393830834
R2: 0.9184495092368911


In [17]:
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='r2')

print("Cross Validation R2:", cv_scores.mean())

Cross Validation R2: 0.9096583875430388


### **SAVE MODEL**
-----

In [18]:
import os

model_dir = r"D:\medical_cost_prediction\Main Folder\models"
os.makedirs(model_dir, exist_ok=True)
model_path = os.path.join(model_dir, "final_model.pkl")
joblib.dump(pipeline, model_path)

print(f"Model saved successfully at: {model_path}")

Model saved successfully at: D:\medical_cost_prediction\Main Folder\models\final_model.pkl


In [19]:
# Define full path
model_path = os.path.join(model_dir, "feature_columns.pkl")
# Save feature column names
joblib.dump(X_train.columns.tolist(), model_path)

print("Feature columns saved at:", model_path)

Feature columns saved at: D:\medical_cost_prediction\Main Folder\models\feature_columns.pkl


#### **SAMPLE PREDICTION**

----

In [20]:
import joblib
# Load trained model
model = joblib.load(r"D:\medical_cost_prediction\Main Folder\models\final_model.pkl")

In [21]:
# Example user input (RAW format — same as original dataset)
new_data = pd.DataFrame({
    "age": [45],
    "sex": ["male"],
    "bmi": [28.5],
    "children": [2],
    "smoker": ["no"]
})
new_data["age_bmi_interaction"] = new_data["age"] * new_data["bmi"]
new_data["sex"] = new_data["sex"].map({"female": 0, "male": 1})
new_data["smoker"] = new_data["smoker"].map({"no": 0, "yes": 1})

if new_data[["sex", "smoker"]].isnull().values.any():
    raise ValueError("Invalid category found in input data")

In [22]:
X_train_sample_pred = X_train
new_data = new_data.reindex(columns=X_train_sample_pred.columns, fill_value=0)

In [23]:
prediction = model.predict(new_data)

print("Predicted Medical Cost (USD):", prediction[0])

Predicted Medical Cost (USD): 22730.6558175357
